In [1]:
pip install sentence-transformers faiss-cpu numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 46.0 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import pickle

# --- Load chunks ---
path = "/kaggle/input/datasets/chunks.jsonl"

chunks = []
with open(path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))

print(f"Loaded {len(chunks)} chunks")

# --- Embed chunks ---
model = SentenceTransformer('all-mpnet-base-v2') 

texts = [c['text'] for c in chunks]
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"Embeddings shape: {embeddings.shape}")

# --- Build FAISS index ---
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype(np.float32))

print(f"FAISS index built with {index.ntotal} vectors")

# --- Save to Kaggle working directory ---
faiss.write_index(index, "/kaggle/working/microfluidics.index")

with open("/kaggle/working/chunk_metadata.pkl", 'wb') as f:
    pickle.dump(chunks, f)

print("Saved index and metadata")

Loaded 903 chunks


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/29 [00:00<?, ?it/s]

Embeddings shape: (903, 768)
FAISS index built with 903 vectors
Saved index and metadata


In [ ]:
def search(query, k=4):
    query_emb = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    scores, indices = index.search(query_emb.astype(np.float32), k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks[idx]
        results.append({
            'score': float(score),
            'chunk_id': chunk['chunk_id'],
            'paper_title': chunk['paper_title'],
            'text': chunk['text'][:200] + '...'
        })
    return results

# Testing whether the query hallucinates or not
results = search("What is the Van den Bergh reaction and how is it used to detect bilirubin?")
for r in results:
    print(f"[{r['score']:.3f}] {r['paper_title']}")
    print(r['text'])
    print()

[0.534] KINETICS OF THE FORMATION OF AZOBILIRUBIN
Bilirubin. The bilirubin used was obtained from Hoffmann-La Roche and was recrystallized from chloroform. It contained no biliverdin and was pure, as shown by its nitrogen content, its equivalent weig...

[0.498] KINETICS OF THE FORMATION OF AZOBILIRUBIN
The coupling reaction between diazotized sulphanilic acid (p-diazobenzenesulphonic acid) and bilirubin has attracted a great deal of attention, because it is used in clinical chemistry for the determi...

[0.496] KINETICS OF THE FORMATION OF AZOBILIRUBIN
ished: a small amount of biliverdin, 2 azodyes, and a yellow product, probably the hydroxypyrromethene carbinol. Biliverdin (an oxidation product of bilirubin) could be adsorbed on a chromatographic ....

[0.494] KINETICS OF THE FORMATION OF AZOBILIRUBIN
ich one molecule of the diazonium salt is coupled to one molecule of bilirubin. The main product of the reaction is a compound containing two molecules of the diazonium salt per molecul